In [ ]:
# ============ ELSA STANDALONE RUNNER — OPT-125M, paper recetesi (Colab, A100/T4) ============
# Sadece Elsa: egitim + WikiText-2 eval. AGORA yok. Icerik:
#   * Resmi repo (LOG-postech/elsa) + 3 yama:
#       1) lib/utils.py z_metric girinti fix'i (release regresyonu; momentum projection fp32-adam'da calissin)
#       2) main.py use_fast=True (tokenizasyon ~10x hizli; fast==slow esitligi asagida assert'li)
#       3) lib/data.py RAM fix (32k ornekte from_list ~7GB Python-int -> from_dict numpy, ~1.6GB; ayni RNG sirasi)
#   * SPARSITY sec -> paper recetesi otomatik (Tablo 5/6, arXiv 2510.01650 LaTeX kaynagindan dogrulanmis)
#   * GPU-adaptif: A100 -> batch2/no-ckpt/bf16 (tam paper recetesi); T4 -> batch1+ckpt+fp16 (kanitli)
#   * Drive persistence: checkpoint (sparsity-scoped klasor) + dataset cache (32k ornek, ~40dk tokenizasyon)
#   * Dogrulanmis reproduksiyon: @90% -> 97.10 (paper 95.33, +1.9%) | @85% -> 75.34 (paper'da yok)
import time, os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
_T0 = time.time()
def log(msg):    print(f"[{time.time()-_T0:7.1f}s] {msg}", flush=True)
def banner(txt): print("\n" + "="*72 + f"\n=== {txt}\n" + "="*72, flush=True)

# ---- KONFIG ----
SPARSITY = 0.85        # 0.5 / 0.6 / 0.7 / 0.8 / 0.85 / 0.9 — recete otomatik kurulur
DRIVE = "/content/drive/MyDrive/agora"   # bos "" = Drive kullanma
# Paper recetesi (Tablo 5: eta/lambda/schedule; Tablo 6: BS/steps; 80-90 hucreleri birlesik -> 0.85'i kapsar)
RECIPES = {  # sparsity: (lr, lmda, schedule, efektif_BS, steps)
    0.5:  (1e-4, 1e-4, "constant", 16, 2048),
    0.6:  (2e-4, 1e-3, "constant", 16, 2048),
    0.7:  (1e-4, 2e-3, "cosine",    8, 4096),
    0.8:  (2e-4, 1e-3, "cosine",    8, 4096),
    0.85: (2e-4, 1e-3, "cosine",    8, 4096),   # 80-90 birlesik hucrelerden
    0.9:  (2e-4, 1e-3, "cosine",    8, 4096),
}
PAPER_WIKI = {0.5: 37.68, 0.6: 41.5, 0.7: 49.57, 0.8: 65.30, 0.9: 95.33}   # Tablo 10 (sadakat referansi)
assert SPARSITY in RECIPES, f"SPARSITY {SPARSITY} icin recete tanimli degil"
ELSA_LR, ELSA_LMDA, LMDA_SCHED, EFF_BS, STEPS = RECIPES[SPARSITY]
banner(f"ELSA @{SPARSITY:.0%}: lr={ELSA_LR:g} lambda={ELSA_LMDA:g}({LMDA_SCHED}) eff-batch={EFF_BS} steps={STEPS}")

# --- 1) deps + yamalar ---
banner("1) DEPS + YAMALAR (~1-2 dk)")
!test -d elsa || git clone https://github.com/LOG-postech/elsa
import pathlib as _pl
_u = _pl.Path("elsa/lib/utils.py")
_lines = _u.read_text().splitlines()
_nfix = 0
for _i, _ln in enumerate(_lines):
    if _ln.strip() == "z_metric = a * (weight**2)":
        _lines[_i] = "        z_metric = a * (weight**2)"; _nfix += 1
_u.write_text("\n".join(_lines) + "\n")
print(f"[patch 1] z_metric girinti duzeltildi ({_nfix} satir)")
!sed -i 's/use_fast=False/use_fast=True/' elsa/main.py
print("[patch 2] tokenizer fast'a alindi")
_d = _pl.Path("elsa/lib/data.py")
_src = _d.read_text()
if "Dataset.from_dict" not in _src:
    _dl = _src.splitlines()
    _i0 = next(i for i, l in enumerate(_dl) if "processed_samples = []" in l)
    _i1 = next(i for i, l in enumerate(_dl) if "return Dataset.from_list(processed_samples)" in l)
    _new = '''    import numpy as _np
    is_gemma_tokenizer = 'gemma' in tokenizer.__class__.__name__.lower()
    ids = _np.empty((nsamples, seqlen), dtype=_np.int64)
    for _j in tqdm(range(nsamples), desc="Generating samples"):
        token_source = random.choice(all_tokens)
        slice_len = seqlen - 1 if is_gemma_tokenizer else seqlen
        start_index = random.randint(0, token_source.shape[1] - slice_len - 1)
        inp = token_source[0, start_index:start_index + slice_len]
        if is_gemma_tokenizer:
            inp = torch.cat([torch.tensor([tokenizer.bos_token_id]), inp])
        ids[_j] = inp.numpy()
    return Dataset.from_dict({"input_ids": ids,
                              "attention_mask": _np.ones_like(ids),
                              "labels": ids.copy()})'''
    _dl[_i0:_i1 + 1] = _new.splitlines()
    _d.write_text("\n".join(_dl) + "\n")
    print("[patch 3] data.py RAM fix uygulandi")
else:
    print("[patch 3] data.py zaten patchli")
!pip -q install "transformers==4.45.0" "tokenizers==0.20.3" "accelerate==1.7.0" \
                "datasets==3.6.0" "huggingface_hub==0.31.2" "safetensors>=0.4.3" \
                "fsspec<=2025.3.0" absl-py wandb einops sentencepiece zstandard
!pip -q install --no-deps "torchao==0.13.0"
log("deps bitti")

# --- 2) surum guard ---
banner("2) SURUM KONTROL")
import huggingface_hub, transformers, datasets
log(f"transformers={transformers.__version__} | datasets={datasets.__version__} | hub={huggingface_hub.__version__}")
if huggingface_hub.__version__ != "0.31.2":
    log("!! hub yanlis surumde. >>> Runtime > Restart session, hucreyi TEKRAR calistir <<<")
    raise SystemExit
log("surumler OK")

# --- 3) GPU + auth + Drive ---
banner("3) GPU + AUTH + DRIVE")
import torch, glob, math, shutil
if not torch.cuda.is_available():
    log("!! CUDA YOK. Runtime > Change runtime type > GPU sec, restart, tekrar kos"); raise SystemExit
p = torch.cuda.get_device_properties(0)
BIG_GPU = p.total_memory > 20e9
prec = "bf16" if torch.cuda.is_bf16_supported() else "fp16"
log(f"GPU: {p.name} | {p.total_memory/1e9:.1f}GB | {prec} | "
    f"{'batch2/no-ckpt' if BIG_GPU else 'batch1+ckpt (T4-guvenli)'}")
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None
assert HF_TOKEN, "HF_TOKEN yok: Colab > Secrets > HF_TOKEN ekle (notebook access acik)"
os.environ["HF_TOKEN"] = HF_TOKEN; os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
from huggingface_hub import login
try:
    login(token=HF_TOKEN, add_to_git_credential=False); log("HF auth OK")
except Exception as e:
    log(f"HF auth FAILED (public modeller yine calisir): {e}")
if DRIVE:
    from google.colab import drive as _gd
    _gd.mount("/content/drive"); os.makedirs(DRIVE, exist_ok=True)
    log(f"Drive hazir: {DRIVE}")

# --- 4) Elsa egitimi ---
banner(f"4) ELSA @{SPARSITY:.0%} egitimi (A100 ~40-90dk | T4 ~3-4.5 saat)")
import subprocess
ELSA_BS, ELSA_ACCUM = (2, EFF_BS // 2) if BIG_GPU else (1, EFF_BS)
ELSA_CKPT = not BIG_GPU
if DRIVE and os.path.isdir(f"{DRIVE}/elsa_dataset") and not os.path.isdir("elsa/dataset"):
    shutil.copytree(f"{DRIVE}/elsa_dataset", "elsa/dataset")
    log("dataset cache Drive'dan yuklendi (~40dk tokenizasyon atlanir)")
!rm -rf elsa_pruned
if LMDA_SCHED == "cosine":
    _lmda_flags = (f"--admm_lmda={ELSA_LMDA} --admm_lmda_schedule_mode=cosine "
                   f"--admm_init_lmda=0.0 --admm_final_lmda={ELSA_LMDA} ")
else:
    _lmda_flags = f"--admm_lmda={ELSA_LMDA} --admm_lmda_schedule_mode=constant "
cmd = (f"python main.py --model=facebook/opt-125m --seqlen=2048 --dataset=c4 "
       f"--sparsity_ratio={SPARSITY} --sparsity_type=unstructured "
       f"--admm_steps={STEPS} --admm_batch_size={ELSA_BS} --admm_gradient_accumulation_steps={ELSA_ACCUM} "
       f"--admm_gradient_checkpointing={ELSA_CKPT} "
       f"--admm_lr={ELSA_LR} --admm_interval=32 --admm_projection_mode=momentum "
       f"--admm_beta1=0.9 --admm_beta2=0.999 " + _lmda_flags +
       f"--admm_logging_steps=32 --admm_eval_steps=1024 --admm_save_inputs=True "
       f"--admm_precision={prec} --eval_zero_shot=False "
       f"--save_model=True --admm_save_path=../elsa_pruned --seed=0")
log("komut: " + cmd)
_te = time.time()
proc = subprocess.Popen(f"cd elsa && {cmd}", shell=True,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
log(f"Elsa bitti | exit_code={proc.returncode} | sure {(time.time()-_te)/60:.1f}dk")
if proc.returncode in (-9, 137):
    log("!! SIGKILL = RAM OOM isareti (patch 3 calisti mi?). GPU OOM ise T4 modu zaten aktif olmali.")
if DRIVE and os.path.isdir("elsa/dataset"):
    shutil.copytree("elsa/dataset", f"{DRIVE}/elsa_dataset", dirs_exist_ok=True)
    log("dataset cache Drive'a kopyalandi")
hits = sorted((h for h in glob.glob("elsa_pruned/**/config.json", recursive=True)
               if f"pruned{SPARSITY}" in h), key=os.path.getmtime)
ELSA_DIR = os.path.dirname(hits[-1]) if hits else None
assert ELSA_DIR, "Elsa checkpoint YOK — subprocess ciktisinin son satirlarina bak (Killed/OOM/traceback)"
log(f"checkpoint: {ELSA_DIR}")

# --- 5) eval (WikiText-2 test, seqlen 2048, fp16 autocast) ---
banner("5) EVAL")
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
DEV, EVAL_SEQLEN = "cuda", 2048
tok = AutoTokenizer.from_pretrained("facebook/opt-125m", use_fast=True)
_txt = "\n\n".join(load_dataset("wikitext", "wikitext-2-raw-v1", split="test")["text"])
tok_slow = AutoTokenizer.from_pretrained("facebook/opt-125m", use_fast=False)
assert tok(_txt).input_ids == tok_slow(_txt).input_ids, "fast != slow tokenizer!"
log("tokenizer check OK"); del tok_slow
test_ids = tok(_txt, return_tensors="pt").input_ids[0]
N_CHUNKS = len(test_ids) // EVAL_SEQLEN

@torch.no_grad()
def ppl_of(model, tag=""):
    model.eval().to(DEV); nll, nt = 0.0, 0; t = time.time()
    for i in range(0, N_CHUNKS * EVAL_SEQLEN, EVAL_SEQLEN):
        b = test_ids[i:i + EVAL_SEQLEN].unsqueeze(0).to(DEV)
        with torch.autocast("cuda", dtype=torch.float16):
            nll += model(b, labels=b).loss.float().item() * (EVAL_SEQLEN - 1)
        nt += EVAL_SEQLEN - 1
    ppl = math.exp(nll / nt)
    log(f"  [eval {tag}] {N_CHUNKS} chunk, {time.time()-t:.0f}s -> PPL {ppl:.2f}")
    return ppl

dense = AutoModelForCausalLM.from_pretrained("facebook/opt-125m", torch_dtype=torch.float32)
PPL_DENSE = ppl_of(dense, "dense"); del dense; torch.cuda.empty_cache()
elsa_m = AutoModelForCausalLM.from_pretrained(ELSA_DIR, torch_dtype=torch.float32)
tot = nz = 0
for n, pr in elsa_m.named_parameters():
    if "decoder.layers" in n and pr.dim() == 2:
        tot += pr.numel(); nz += (pr != 0).sum().item()
_real_sp = 1 - nz / tot
assert abs(_real_sp - SPARSITY) <= 0.01, f"checkpoint sparsity {_real_sp:.3f} != hedef {SPARSITY}"
PPL_ELSA = ppl_of(elsa_m, "elsa"); del elsa_m; torch.cuda.empty_cache()
if DRIVE:
    _dst = f"{DRIVE}/elsa_pruned_{int(SPARSITY*100)}"
    shutil.copytree("elsa_pruned", _dst, dirs_exist_ok=True)
    log(f"checkpoint Drive'a kopyalandi: {_dst}")

banner("SONUC")
print(f"dense fp16 referans : PPL {PPL_DENSE:8.2f}  (paper 27.65)")
_pw = PAPER_WIKI.get(SPARSITY)
if _pw:
    print(f"Elsa @{SPARSITY:.0%}          : PPL {PPL_ELSA:8.2f}  (paper Tablo-10: {_pw} | sapma {100*(PPL_ELSA/_pw-1):+.1f}%)")
else:
    print(f"Elsa @{SPARSITY:.0%}         : PPL {PPL_ELSA:8.2f}  (paper'da bu seviye yok)")
print(f"sparsity: {_real_sp:.3f} | recete: lr={ELSA_LR:g} lambda={ELSA_LMDA:g}({LMDA_SCHED}) "
      f"eff-batch={EFF_BS} steps={STEPS} | {p.name.split()[0]} {prec}-train / fp16-eval")
log(f"BITTI | toplam {(time.time()-_T0)/60:.1f} dk")